In [ ]:
# ============================================================================
# JOIN TEMPORAL DE PADRÕES HEXAGONAIS
# ============================================================================
# Problema identificado:
#   Class8590 descreve o padrão da região em 1985-1990 — pode estar
#   desatualizado para pixels com T=2010 ou T=2015.
#
# Solução:
#   Para cada pixel, selecionar o padrão do período mais próximo
#   e anterior a T — o estado da região quando o pixel entrou em pastagem.
#
# Períodos disponíveis no shapefile:
#   Class8590: 1985-1990  → usar se T <= 1990
#   Class9095: 1990-1995  → usar se 1990 < T <= 1995
#   Class9500: 1995-2000  → usar se 1995 < T <= 2000
#   Class0005: 2000-2005  → usar se 2000 < T <= 2005
#   Class0510: 2005-2010  → usar se 2005 < T <= 2010
#   Class1015: 2010-2015  → usar se 2010 < T <= 2015
#   Class1520: 2015-2020  → usar se T > 2015
#
# Output:
#   dataset_v2 com coluna 'pattern_T' — padrão contemporâneo a T
#   e 'pattern_period' — qual período foi usado
# ============================================================================

import os
import struct
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

# ── Caminhos ────────────────────────────────────────────────────────────────
BASE_DIR  = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
OUT_DIR   = BASE_DIR / "dataset_v2"
SHP_PATH  = r"D:\Projetos\Cerrado\hex_Cerrado_class_Change.shp"
DBF_PATH  = r"D:\Projetos\Cerrado\hex_Cerrado_class_Change.dbf"

# Mapeamento período → coluna
PERIOD_MAP = [
    (1990, 'Class8590'),   # T <= 1990
    (1995, 'Class9095'),   # 1990 < T <= 1995
    (2000, 'Class9500'),   # 1995 < T <= 2000
    (2005, 'Class0005'),   # 2000 < T <= 2005
    (2010, 'Class0510'),   # 2005 < T <= 2010
    (2015, 'Class1015'),   # 2010 < T <= 2015
    (9999, 'Class1520'),   # T > 2015
]

def get_pattern_col(T):
    """Retorna (coluna, período_label) para um dado T."""
    for cutoff, col in PERIOD_MAP:
        if T <= cutoff:
            return col, col.replace('Class', '')
    return 'Class1520', '1520'

print("✅ Configuração OK")
print(f"   Dataset v2: {OUT_DIR}")
print()
print("Mapeamento T → coluna de padrão:")
print(f"  {'T':<8} {'Coluna':<12} {'Período'}")
print("-" * 35)
test_Ts = [1987, 1992, 1998, 2003, 2008, 2013, 2018]
for t in test_Ts:
    col, per = get_pattern_col(t)
    print(f"  {t:<8} {col:<12} {per}")

In [ ]:
# ============================================================================
# CARREGAR SHAPEFILE — todos os períodos
# ============================================================================

def read_shp_centroids(path):
    ids, cxs, cys = [], [], []
    with open(path, 'rb') as f:
        f.read(100)
        while True:
            hdr = f.read(8)
            if len(hdr) < 8: break
            rec_num = struct.unpack('>I', hdr[:4])[0]
            clen    = struct.unpack('>I', hdr[4:])[0]
            data    = f.read(clen * 2)
            if len(data) < 4: break
            stype = struct.unpack('<I', data[:4])[0]
            if stype == 0: continue
            bbox = struct.unpack('<4d', data[4:36])
            ids.append(rec_num)
            cxs.append((bbox[0] + bbox[2]) / 2)
            cys.append((bbox[1] + bbox[3]) / 2)
    return np.array(ids), np.array(cxs), np.array(cys)


def read_dbf(path):
    with open(path, 'rb') as f:
        f.read(4)
        n_recs = struct.unpack('<I', f.read(4))[0]
        hsize  = struct.unpack('<H', f.read(2))[0]
        f.read(22)
        fields = []
        while True:
            fd = f.read(32)
            if fd[0] == 0x0D: break
            name   = fd[:11].replace(b'\x00', b'').decode('latin-1').strip()
            ftype  = chr(fd[11])
            length = fd[16]
            fields.append((name, ftype, length))
        f.seek(hsize)
        records = []
        for _ in range(n_recs):
            d = f.read(1)
            if d == b'*':
                for _, _, l in fields: f.read(l)
                continue
            row = {}
            for name, ftype, length in fields:
                raw = f.read(length).decode('latin-1').strip()
                if ftype == 'N' and raw and '.' not in raw:
                    row[name] = int(raw)
                else:
                    row[name] = raw if raw else None
            records.append(row)
    return pd.DataFrame(records)


print("Carregando shapefile hexagonal...")
ids, cxs, cys = read_shp_centroids(SHP_PATH)
dbf = read_dbf(DBF_PATH)
dbf['hex_id'] = dbf['OBJECTID_1'].astype(int)

# Todas as colunas de padrão
pattern_cols = ['Class8590', 'Class9095', 'Class9500',
                'Class0005', 'Class0510', 'Class1015', 'Class1520']

hex_df = pd.DataFrame({'hex_id': ids, 'cx': cxs, 'cy': cys})
hex_df = hex_df.merge(
    dbf[['hex_id'] + pattern_cols], on='hex_id', how='left')

print(f"✅ {len(hex_df):,} hexágonos carregados")
print(f"   Colunas de padrão: {pattern_cols}")
print()

# Verificar cobertura de cada período
print("Preenchimento por período:")
for col in pattern_cols:
    n_valid = hex_df[col].notna().sum()
    print(f"   {col}: {n_valid:,}/{len(hex_df):,} hexágonos com padrão")

In [ ]:
# ============================================================================
# PROJEÇÃO E ATRIBUIÇÃO DE HEXÁGONO
# ============================================================================

# Parâmetros da projeção Albers (mesmos do spatial_split.py)
LEFT   = -60.47269846482955
RIGHT  = -41.277407642235204
TOP    =  -2.3319366460458646
BOTTOM = -24.681931083411143
NROWS  = 82933
NCOLS  = 71227

a = 6378160.0; f = 1/298.25; b = a*(1-f)
e2 = 1-(b/a)**2; e = np.sqrt(e2)
lon0 = np.radians(-60.0); lat0 = np.radians(-32.0)
phi1 = np.radians(-5.0);  phi2 = np.radians(-42.0)

def alpha(phi):
    sp = np.sin(phi)
    return (1-e2)*(sp/(1-e2*sp**2) - np.log((1-e*sp)/(1+e*sp))/(2*e))

def Nf(phi): return np.cos(phi)/np.sqrt(1-e2*np.sin(phi)**2)

n  = (Nf(phi1)**2 - Nf(phi2)**2) / (2*(alpha(phi2)-alpha(phi1)))
C  = Nf(phi1)**2 + n*alpha(phi1)
r0 = a*np.sqrt(C - n*alpha(lat0)) / n

def rowcol_to_albers(row, col):
    lon = LEFT + (np.asarray(col, float) + 0.5)*(RIGHT-LEFT)/NCOLS
    lat = TOP  + (np.asarray(row, float) + 0.5)*(BOTTOM-TOP)/NROWS
    phi = np.radians(lat); lam = np.radians(lon)
    r   = a * np.sqrt(np.maximum(C - n*alpha(phi), 0)) / n
    th  = n * (lam - lon0)
    return r*np.sin(th), r0 - r*np.cos(th)

print("✅ Projeção Albers configurada")

In [ ]:
# ============================================================================
# JOIN TEMPORAL — atribuir padrão contemporâneo a T
# ============================================================================

# Carregar dataset v2
df = pd.read_csv(OUT_DIR / "dataset_v2_full.csv")
df = df[df['label'].notna()].reset_index(drop=True)
df['label'] = df['label'].astype(int)
df['T']     = df['T'].astype(int)

print(f"Dataset v2: {len(df):,} pixels")
print(f"Colunas disponíveis: {df.columns.tolist()}")

# Converter pixels para Albers
print("\nConvertendo coordenadas...")
px, py = rowcol_to_albers(df['row'].values, df['col'].values)

# Encontrar hexágono mais próximo (mesmo do spatial_split.py)
HX = hex_df['cx'].values
HY = hex_df['cy'].values

print("Atribuindo hexágonos...")
BATCH = 500
assigned = []
for i in tqdm(range(0, len(px), BATCH), desc="Nearest hex"):
    dx = px[i:i+BATCH, None] - HX[None, :]
    dy = py[i:i+BATCH, None] - HY[None, :]
    assigned.extend(np.argmin(dx**2 + dy**2, axis=1).tolist())

df['hex_idx'] = assigned

# Para cada pixel, selecionar padrão contemporâneo a T
print("\nAtribuindo padrão contemporâneo a T...")

pattern_T   = []
period_used = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Pattern join"):
    T       = int(row['T'])
    hex_idx = int(row['hex_idx'])
    col_name, period = get_pattern_col(T)

    pat = hex_df.iloc[hex_idx][col_name]

    # Fallback: se padrão ausente nesse período, usar período anterior
    if pd.isna(pat) or pat == '':
        # Tentar períodos anteriores
        for cutoff, fallback_col in PERIOD_MAP:
            fallback_pat = hex_df.iloc[hex_idx][fallback_col]
            if pd.notna(fallback_pat) and fallback_pat != '':
                pat = fallback_pat
                col_name = fallback_col
                period = fallback_col.replace('Class', '') + '_fallback'
                break

    pattern_T.append(pat)
    period_used.append(period)

df['pattern_T']      = pattern_T
df['pattern_period'] = period_used

print(f"\n✅ Join temporal concluído!")
print(f"\nDistribuição de períodos utilizados:")
print(df['pattern_period'].value_counts().to_string())
print(f"\nFallbacks utilizados: "
      f"{df['pattern_period'].str.contains('fallback').sum()}")

In [ ]:
# ============================================================================
# COMPARAÇÃO: Class8590 vs pattern_T
# ============================================================================
# Quantos pixels têm padrão diferente quando usamos o período correto?

import matplotlib.pyplot as plt

# Verificar concordância entre Class8590 e pattern_T
if 'Class8590' in df.columns:
    concordancia = (df['Class8590'] == df['pattern_T']).mean()
    print(f"Concordância Class8590 vs pattern_T: {concordancia:.1%}")
    print(f"Pixels com padrão DIFERENTE: "
          f"{(df['Class8590'] != df['pattern_T']).sum():,} "
          f"({100*(df['Class8590'] != df['pattern_T']).mean():.1f}%)")
    print()

# Função de classificação CC*/CD*/NC
def grupo_padrao(p):
    s = str(p)
    if s.startswith('CC'): return 'CC* (cluster)'
    if s.startswith('CD'): return 'CD* (disperso)'
    if s == 'NC':          return 'NC'
    return 'Outro'

df['grupo_T']    = df['pattern_T'].apply(grupo_padrao)
if 'Class8590' in df.columns:
    df['grupo_8590'] = df['Class8590'].apply(grupo_padrao)

# Hipótese com pattern_T
cc_T  = df[df['grupo_T'] == 'CC* (cluster)']['label']
cd_T  = df[df['grupo_T'] == 'CD* (disperso)']['label']
nc_T  = df[df['grupo_T'] == 'NC']['label']

from scipy import stats as sp_stats

print("Hipótese CC* → Rápido com pattern_T (contemporâneo):")
print(f"  CC*: {cc_T.mean()*100:.1f}% rápido (n={len(cc_T):,})")
print(f"  CD*: {cd_T.mean()*100:.1f}% rápido (n={len(cd_T):,})")
print(f"  NC:  {nc_T.mean()*100:.1f}% rápido (n={len(nc_T):,})")

if len(cc_T) > 0 and len(cd_T) > 0:
    obs = np.array([
        [cc_T.sum(), len(cc_T) - cc_T.sum()],
        [cd_T.sum(), len(cd_T) - cd_T.sum()]
    ])
    _, p_val, _, _ = sp_stats.chi2_contingency(obs)
    diff = (cc_T.mean() - cd_T.mean()) * 100
    h = 2*(np.arcsin(np.sqrt(cc_T.mean())) -
           np.arcsin(np.sqrt(cd_T.mean())))
    print(f"  Diferença CC*-CD*: {diff:+.1f}pp  "
          f"p={p_val:.4f}  Cohen's h={h:.3f}")

    if 'Class8590' in df.columns:
        cc_8590 = df[df['grupo_8590']=='CC* (cluster)']['label']
        cd_8590 = df[df['grupo_8590']=='CD* (disperso)']['label']
        diff_old = (cc_8590.mean() - cd_8590.mean()) * 100
        print(f"\n  Comparação com Class8590 (antigo):")
        print(f"    Class8590: diferença={diff_old:+.1f}pp")
        print(f"    pattern_T: diferença={diff:+.1f}pp")
        if abs(diff) > abs(diff_old):
            print(f"    ✅ pattern_T tem separação MAIOR — mais discriminativo")
        else:
            print(f"    ⚠️  Class8590 tinha separação similar ou maior")

In [ ]:
# ============================================================================
# SALVAR DATASET v2 COM PADRÃO TEMPORAL
# ============================================================================

import matplotlib.pyplot as plt

# Colunas finais
cols_model = ['row', 'col', 'T', 'label', 'hex_id',
              'Class8590', 'pattern_T', 'pattern_period',
              'grupo_T', 'split']
cols_out = [c for c in cols_model if c in df.columns]

# Salvar dataset completo atualizado
df[cols_out].to_csv(OUT_DIR / "dataset_v2_full.csv", index=False)
print(f"✅ dataset_v2_full.csv atualizado com pattern_T")

# Salvar splits
for split in ['train', 'val', 'test']:
    df_split = df[df['split'] == split][cols_out].reset_index(drop=True)
    df_split.to_csv(OUT_DIR / f"dataset_v2_{split}.csv", index=False)
    n1 = (df_split['label']==1).sum()
    cc_pct = (df_split['grupo_T']=='CC* (cluster)').mean()*100
    print(f"✅ {split}: {len(df_split):,} pixels | "
          f"label=1: {100*n1/len(df_split):.1f}% | "
          f"CC*: {cc_pct:.1f}%")

# Gráfico comparativo Class8590 vs pattern_T
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col_grupo, titulo in [
    (axes[0], 'grupo_8590', 'Class8590 (1985-1990, estático)'),
    (axes[1], 'grupo_T',    'pattern_T (contemporâneo a T)'),
]:
    if col_grupo not in df.columns: continue
    stats = df.groupby(col_grupo).agg(
        n          = ('label','count'),
        pct_rapido = ('label','mean')
    ).reset_index().sort_values('pct_rapido', ascending=True)

    cores = ['#c0392b' if 'CC' in g else '#2980b9' if 'CD' in g
             else '#7f8c8d' for g in stats[col_grupo]]
    ax.barh(stats[col_grupo], stats['pct_rapido']*100, color=cores)
    ax.axvline(x=df['label'].mean()*100, color='black',
               linestyle='--', alpha=0.5, label='Média geral')
    ax.set_xlabel('% conversão rápida (label=1)')
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlim(0, 100)
    for i, (_, r) in enumerate(stats.iterrows()):
        ax.text(r['pct_rapido']*100 + 1, i,
                f"{r['pct_rapido']*100:.1f}% (n={r['n']:,})",
                va='center', fontsize=9)
    ax.legend()

plt.suptitle('Comparação: padrão estático vs contemporâneo a T',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUT_DIR / 'pattern_comparison_static_vs_temporal.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'='*60}")
print(f"DATASET v2 ATUALIZADO COM PADRÃO TEMPORAL")
print(f"{'='*60}")
print(f"\nColuna de routing: 'pattern_T'")
print(f"  CC* → Canal Rápido")
print(f"  CD* + NC → Canal Lento")
print(f"\n🎯 Próximo passo: arquitetura de dois canais com routing")
print(f"   determinístico por pattern_T + gate suave aprendido")